# Effective Data Visualization with Python
**Hands-on notebook** · UCF Office of Research Cyberinfrastructure · October 29, 2026

In this notebook, we turn the design principles from the slides into code. We use one dataset and three libraries. For each library, we:

1. make the simplest possible plot,
2. improve it step by step,
3. see what the library does best, and
4. practice with short exercises.

| Part | Library | Main idea |
|---|---|---|
| 1 | Matplotlib | Build a figure piece by piece, with full control |
| 2 | Seaborn | Statistical plots straight from a table |
| 3 | Plotly | Interactive plots you can explore and share |

## Getting started

- Click a code cell and press **Shift + Enter** to run it. Run the cells from top to bottom.
- `[*]` next to a cell means it is still running.
- You can edit any cell and run it again.

In [ ]:
import pandas as pd                # tables of data (DataFrames)
import matplotlib.pyplot as plt    # Part 1
import seaborn as sns              # Part 2
import plotly.express as px        # Part 3

import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook"   # display setting so Plotly charts appear in any Jupyter setup

### The data: Palmer penguins

Each row is one adult penguin measured near Palmer Station, Antarctica, between 2007 and 2009: 333 penguins from three species.

In [ ]:
penguins = pd.read_csv("data/penguins.csv")
penguins.head()

| Column | Meaning |
|---|---|
| `species` | Adelie, Chinstrap, or Gentoo |
| `island` | Island where the penguin was measured |
| `bill_length_mm`, `bill_depth_mm` | Bill length and bill depth (height), in millimeters ([diagram](https://allisonhorst.github.io/palmerpenguins/)) |
| `flipper_length_mm` | Flipper (wing) length, in millimeters |
| `body_mass_g` | Body mass, in grams |
| `sex` | female or male |
| `year` | Year the penguin was measured |

`penguins` is a pandas **DataFrame**, a table. `penguins["body_mass_g"]` selects one column.

*Data: Gorman, Williams & Fraser (2014), distributed in the palmerpenguins package (CC0).*

---
## Part 1. Matplotlib: full control

Matplotlib is Python's core plotting library. You build a figure piece by piece, so you can control every detail. Seaborn (Part 2) is built on top of it.

Every Matplotlib plot in this notebook follows the same recipe ([anatomy of a figure](https://matplotlib.org/stable/gallery/showcase/anatomy.html)):

```python
fig, ax = plt.subplots()    # 1. create a figure (the canvas) with axes (the plot area)
ax.scatter(x, y)            # 2. draw data on the axes
ax.set_xlabel("...")        # 3. customize: labels, title, colors, ...
plt.show()                  # 4. display the figure
```

### 1.1 Basic Scatter Plot

**Question:** Do penguins with longer flippers weigh more?

**Plan:** Flipper length and body mass are both numbers, so we use a **scatter plot**: one dot per penguin.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(penguins["flipper_length_mm"], penguins["body_mass_g"])
plt.show()

The first column goes on the x-axis, the second on the y-axis. The dots rise from left to right: penguins with longer flippers tend to be heavier.

**Change the canvas size.** `figsize=(width, height)` inside `plt.subplots()` sets the size of the figure, in inches.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))    # new: 8 inches wide, 4 inches tall
ax.scatter(penguins["flipper_length_mm"], penguins["body_mass_g"])
plt.show()

**Try it:** change the two numbers and run the cell again.

The plot now has more room, but someone who has not seen our code still cannot tell what the axes show or what units they use.

### 1.2 Improve it

**Step 1: Explain.** Label both axes with units, and use a title that states the finding. New lines are marked `# new`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(penguins["flipper_length_mm"], penguins["body_mass_g"])
ax.set_xlabel("Flipper length (mm)")                                 # new
ax.set_ylabel("Body mass (g)")                                       # new
ax.set_title("Penguins with longer flippers tend to be heavier")     # new
plt.show()

**Step 2: Make it readable.** Many dots overlap. Smaller, partly transparent dots show where they pile up.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(
    penguins["flipper_length_mm"],
    penguins["body_mass_g"],
    s=20,            # new: dot size
    alpha=0.5,       # new: transparency, from 0 (invisible) to 1 (solid)
    color="teal",    # new: a color name, or a hex code such as "#1f77b4"
)
ax.set_xlabel("Flipper length (mm)")
ax.set_ylabel("Body mass (g)")
ax.set_title("Penguins with longer flippers tend to be heavier")
plt.show()

**Try it:** change `color`, `s`, or `alpha` and run the cell again.

### 1.3 Same recipe, different chart

**Question:** Which species is the heaviest on average?

**Plan:** We compare one number per species, so we use a **bar chart**. First, we calculate the average body mass of each species.

In [ ]:
mean_mass = penguins.groupby("species")["body_mass_g"].mean()
mean_mass

Read this as: *group the rows by species, take the `body_mass_g` column, and calculate the mean of each group.* The species names are in `mean_mass.index` and the averages are in `mean_mass.values`.

In [ ]:
fig, ax = plt.subplots()
ax.bar(mean_mass.index, mean_mass.values)
ax.set_ylabel("Average body mass (g)")
ax.set_title("Average body mass by species")
plt.show()

Only the drawing method changed: `ax.bar` instead of `ax.scatter`. Other chart types follow the same recipe, for example `ax.plot` (line), `ax.hist` (histogram), and `ax.boxplot`.

Matplotlib starts bars at zero, as it should: the length of a bar represents its value.

### 1.4 Where Matplotlib shines: control over every element

Suppose this chart goes on a slide whose message is *Gentoo penguins are much heavier*. Matplotlib lets us style each element on its own:

- **Sort** the bars so the ranking is immediate, and make them **horizontal** so the names are easy to read.
- **Highlight** one bar and keep the others gray.
- **Label the values** directly, so readers do not have to estimate them from the axis.
- **Remove** borders that add nothing.

In [ ]:
ranked_mass = mean_mass.sort_values()    # smallest to largest; barh draws them bottom to top

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.barh(ranked_mass.index, ranked_mass.values, color="lightgray")
bars[-1].set_color("tab:blue")                    # color only the last (top) bar
ax.bar_label(bars, fmt="{:,.0f} g", padding=4)    # write each value at the end of its bar
ax.spines[["top", "right"]].set_visible(False)    # remove the top and right borders
ax.set_xlabel("Average body mass (g)")
ax.set_title("Gentoo penguins weigh about 1,400 g more than the other species")
plt.show()

`bars` holds the three bars, so we can change any one of them. Element-by-element control like this is what Matplotlib does best.

### 1.5 Save the figure

`fig` still holds the bar chart, so we can save it. Use **PNG** for slides and documents. Use **SVG** or **PDF** for papers: these vector formats stay sharp at any size.

In [ ]:
fig.savefig("average_body_mass.png", dpi=300, bbox_inches="tight")   # dpi sets the resolution
fig.savefig("average_body_mass.svg", bbox_inches="tight")             # "tight" trims extra white space

The files appear in the file browser on the left, next to this notebook.

### Exercise 1: Fix this plot

**Question:** Which group of penguins (species and sex) is the heaviest on average?

A colleague made the chart below to answer this question. It runs, but it breaks several rules from the slides. Run it, then fix it **by editing the existing lines**; you only need to write a few new things.

1. **Trust:** Where does the y-axis start? Do the bar lengths match the values?
2. **Show:** For a "which is heaviest?" question, is the order of the bars helpful? Can you read the group names?
3. **Focus:** Do the six colors mean anything? Does the black grid help?
4. **Explain:** Does the axis label give units? Does the title say anything?
5. **Include:** Is every bar easy to see? Is the figure big enough to read?

<details><summary>Hint</summary>

- Matplotlib starts bars at zero by default.
- Add `.sort_values()` at the end of the `group_mass = ...` line, and use `ax.barh` so long names fit. With horizontal bars the values are on the x-axis, so use `ax.set_xlabel`.
- One color is enough, for example `color="tab:blue"`. Delete the grid line.
- Find a good size for the figure.

</details>

In [ ]:
# Supplied: the average body mass of each group, such as "Adelie female"
group = penguins["species"] + " " + penguins["sex"]
group_mass = penguins.groupby(group)["body_mass_g"].mean()

fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(group_mass.index, group_mass.values,
       color=["red", "orange", "yellow", "lime", "cyan", "magenta"])
ax.set_ylim(3000, 5600)
ax.grid(color="black", linewidth=1)
ax.set_ylabel("body_mass_g")
ax.set_title("Data")
plt.show()

### Exercise 2: Improve a histogram

**Question:** What flipper lengths are typical? Is there one typical length, or more than one?

A **histogram** shows how values are distributed: it splits the values into ranges (bins) and counts the penguins in each. Run the draft below, then improve it:

1. **Explain:** add an x-axis label with units, a y-axis label, and a title.
2. **Choose the bins:** inside `ax.hist(...)`, try `bins=5`, then `bins=40`. Which one hides the shape? Which one is too noisy? Keep a value in between.
3. **Readability:** choose one `color` and add `edgecolor="white"` so neighboring bars are easy to tell apart.
4. **Interpret:** in the comment at the end of the cell, write how many peaks you see.

*Finished early?* Mark the average with `ax.axvline(penguins["flipper_length_mm"].mean(), color="black", linestyle="--")`. Are many penguins close to the average?

<details><summary>Hint</summary>

Settings go inside the parentheses, separated by commas. For example: `ax.hist(penguins["flipper_length_mm"], bins=10, color="teal")`. The labels use the same commands as in 1.2.

</details>

In [ ]:
fig, ax = plt.subplots()
ax.hist(penguins["flipper_length_mm"])

plt.show()

# How many peaks do you see?

### Why Matplotlib?

- **Full control** over every element: individual bars, text, colors, and layout.
- **Publication-quality output** in PNG, SVG, or PDF.
- **The foundation** for Seaborn, so everything from this part also works in Part 2.

The trade-off: plots that split data into groups or calculate statistics need more code. That is where Seaborn helps.

*Tip:* online, you will also see a shorter style without `ax`, such as `plt.scatter(...)` and `plt.xlabel(...)`. It works for a single quick plot. We use `fig, ax` because it also works for multi-panel figures and for Seaborn.

---
## Part 2. Seaborn: statistical plots from a table

Seaborn is built on Matplotlib and designed for statistical graphics. Instead of passing columns one at a time, you give it the whole table and name the columns to use. Seaborn handles the grouping, colors, legend, and statistics.

```python
ax = sns.histplot(data=penguins, x="column_name", hue="group_column")   # draw
ax.set_xlabel("...")   # Seaborn draws on Matplotlib axes, so Part 1 commands still work
plt.show()
```

### 2.1 Basic Seaborn Plot

**Question:** Why did the flipper-length histogram in Exercise 2 have two peaks?

**Plan:** Make the same histogram with Seaborn, then split it by species.

In [ ]:
sns.histplot(data=penguins, x="flipper_length_mm")
plt.show()

The same histogram in one line. We pass the table (`data=`) and a column name (`x=`), and Seaborn labels the axes with the column names.

**Step 1: Split the data into groups with `hue`.**

In [ ]:
sns.histplot(data=penguins, x="flipper_length_mm", hue="species")
plt.show()

`hue="species"` gives each species its own color and adds a legend. In Matplotlib, this would take a separate `ax.hist` call for each species.

Mystery solved: Gentoo penguins have longer flippers and form the second peak.

**Step 2: Accessible colors and clear labels.**

In [ ]:
ax = sns.histplot(
    data=penguins, x="flipper_length_mm", hue="species",
    palette="colorblind",   # new: colors that stay distinct for most types of color blindness
    element="step",         # new: outlined bars, so overlapping groups stay visible
)
ax.set_xlabel("Flipper length (mm)")                         # new: same commands as Part 1
ax.set_ylabel("Number of penguins")                          # new
ax.set_title("Gentoo penguins have the longest flippers")    # new
plt.show()

`sns.histplot` returns the Matplotlib axes (`ax`), so we finish the plot with the commands from Part 1.

### 2.2 Where Seaborn shines: statistics in one line

**Question:** Do penguins with longer bills have shallower bills? (Bill depth is the height of the bill.)

**Plan:** A scatter plot with a trend line. `sns.lmplot` draws the points, fits a straight line (a linear regression), and shades its 95% confidence interval.

In [ ]:
sns.lmplot(data=penguins, x="bill_length_mm", y="bill_depth_mm")
plt.show()

Across all penguins, the line slopes down: longer bills seem to go with shallower bills.

Now add one argument, `hue="species"`, to fit a separate line for each species. **Predict before you run it:** will the three lines slope down too?

In [ ]:
g = sns.lmplot(
    data=penguins, x="bill_length_mm", y="bill_depth_mm",
    hue="species", palette="colorblind",   # new
)
g.set_axis_labels("Bill length (mm)", "Bill depth (mm)")
plt.show()

Within **every** species, the line slopes **up**. The overall downward trend comes from differences between species: Gentoo penguins have long but shallow bills. A trend that reverses when the data are split into groups is called **Simpson's paradox**.

One extra argument made Seaborn fit three regressions, and it changed our conclusion. Before trusting a trend, ask whether groups in the data could be creating or hiding it.

`lmplot` builds a whole figure (it can also make several panels), so it returns a grid object, `g`, that we label with `g.set_axis_labels(...)`.

### Exercise 3: What does the average hide?

**Question:** Within each species, how does body mass compare between female and male penguins?

The draft uses `sns.barplot` to show the same six averages as Exercise 1. Seaborn calculates them for us (no `groupby` needed) and adds a 95% confidence interval (the thin black line). But each bar still shows only one number.

1. **Show the data:** change `sns.barplot` to `sns.boxplot` and run the cell. Then try `sns.violinplot`. Keep the one that best shows how much females and males overlap.
2. **Accessible colors:** add `palette="colorblind"`.
3. **Explain:** label both axes (with units) and add a title.
4. **Interpret:** is every male heavier than every female of the same species? Answer in the comment.

*Finished early?* Try `sns.swarmplot` with the extra setting `dodge=True` to draw every penguin as a dot.

<details><summary>Hint</summary>

In step 1, only the function name changes; all the settings stay the same. In a box plot, the middle line is the median, the box holds the middle 50% of penguins, and the whiskers show the typical range. A violin is wider where values are common.

</details>

In [ ]:
ax = sns.barplot(data=penguins, x="species", y="body_mass_g", hue="sex")

plt.show()

# Is every male heavier than every female of the same species?

### Why Seaborn?

- **Plots from column names:** `hue` splits the data into groups, with colors and a legend.
- **Statistics built in:** bins, means with confidence intervals, regression lines, and box plots.
- **Matplotlib underneath:** Part 1 commands customize the result.

More Seaborn chart types are in the extra section at the end of the notebook.

The trade-off: the result is a static image, so readers cannot ask for details. That is where Plotly helps.

---
## Part 3. Plotly: interactive plots

Plotly makes interactive charts that run in a web browser: readers can hover to see values, zoom in, and show or hide groups. **Plotly Express** (`px`) uses the same style as Seaborn: a table plus column names.

```python
fig = px.scatter(penguins, x="column_name", y="column_name", color="group_column")
fig.update_layout(...)   # optional: change the figure after creating it
fig.show()
```

### 3.1 Basic Plotly Plot

**Question:** Back to Part 1: do penguins with longer flippers weigh more? This time, we also want to explore individual penguins.

**Plan:** The same scatter plot, made interactive.

In [ ]:
fig = px.scatter(penguins, x="flipper_length_mm", y="body_mass_g")
fig.show()

**Try it:**
- **Hover** over a dot to see its values.
- **Drag** a rectangle to zoom in. **Double-click** to zoom back out.
- Click the **camera icon** (top right) to download the chart as a PNG.

### 3.2 Improve it

**Step 1: Show the groups and explain the axes.** The `labels` setting is a dictionary: it pairs each column name with the text readers should see.

In [ ]:
fig = px.scatter(
    penguins, x="flipper_length_mm", y="body_mass_g",
    color="species",       # new: one color per species, with a legend
    symbol="species",      # new: one marker shape per species, so color is not the only cue
    labels={               # new: readable axis titles with units
        "flipper_length_mm": "Flipper length (mm)",
        "body_mass_g": "Body mass (g)",
        "species": "Species",
    },
    title="Longer flippers go with heavier penguins, in every species",   # new
)
fig.show()

**Try it:** click a species in the legend to hide it. Double-click a species to show only that one; double-click it again to show all.

**Step 2: Add details on demand.** Show extra columns on hover, and adjust the look.

In [ ]:
fig = px.scatter(
    penguins, x="flipper_length_mm", y="body_mass_g",
    color="species", symbol="species",
    labels={
        "flipper_length_mm": "Flipper length (mm)",
        "body_mass_g": "Body mass (g)",
        "species": "Species",
    },
    title="Longer flippers go with heavier penguins, in every species",
    hover_data=["island", "sex", "year"],   # new: extra details shown on hover
    opacity=0.7,                             # new: transparency, as in Part 1
    template="plotly_white",                 # new: a cleaner background
)
fig.update_layout(font_size=14)    # new: change the figure after creating it
fig.show()

Hovering now shows each penguin's island, sex, and year without cluttering the chart. Notice that the main message (title, colors, and labels) is visible without hovering: interaction should add detail, not hide the point.

### 3.3 Where Plotly shines

**Question:** In the box plot from Exercise 3, what exactly are the medians, and which penguins are the extremes?

**Plan:** Make the box plot interactive. `points="all"` draws every penguin beside its box.

In [ ]:
fig = px.box(
    penguins, x="species", y="body_mass_g", color="sex",
    points="all",                       # show every penguin next to the boxes
    hover_data=["island", "year"],
    labels={"body_mass_g": "Body mass (g)", "species": "Species", "sex": "Sex"},
    title="Body mass by species and sex",
    template="plotly_white",
)
fig.show()

- Hover over a **box** to read its exact median and quartiles.
- Hover over a **dot** to identify that penguin. Can you find the heaviest Gentoo?

`px.box` takes the same settings as `px.scatter`; most Plotly Express functions do.

**Share it.** An HTML file keeps all the interactivity and opens in any web browser, without Python.

In [ ]:
fig.write_html("body_mass_interactive.html")

To share the file, right-click it in the file browser, choose **Download**, and open it in your web browser or send it to a colleague.

### Exercise 4: Make the message visible

The slides warned that *interactive does not automatically mean better*: the main finding should be visible without hovering.

**Question:** How do bill length and body mass differ among the three species?

The draft is interactive, but every penguin looks the same, so the species pattern is hidden. Improve it:

1. **Show the groups:** color the dots by species and give each species its own marker shape.
2. **Explain:** use `labels=` for readable axis titles with units, and write a `title` that states what you see.
3. **Details on demand:** add `hover_data=["island", "sex"]`.
4. **Explore:** in the legend, double-click **Chinstrap** to show only that species. Hover to find the Chinstrap with the **longest bill**. Record its bill length, sex, and island in the comment.

*Finished early?* Save your chart with `fig.write_html("my_penguin_chart.html")`.

<details><summary>Hint</summary>

Start from the code in Step 2 of 3.2 and change the column names. The keys in `labels` must match the column names exactly, for example `"bill_length_mm": "Bill length (mm)"`.

</details>

In [ ]:
fig = px.scatter(penguins, x="bill_length_mm", y="body_mass_g")

fig.show()

# Longest Chinstrap bill: ... mm, sex: ..., island: ...

### Why Plotly?

- **Exploration:** hover, zoom, and legend clicks let readers find details without cluttering the chart.
- **Sharing:** one HTML file keeps the interactivity and opens in any browser.
- **Simple syntax:** the same table-plus-column-names style as Seaborn.

The trade-off: for a printed paper, a static Matplotlib or Seaborn figure is usually simpler to fine-tune.

---
## Wrap-up

### The same scatter plot in three libraries

| | Matplotlib | Seaborn | Plotly Express |
|---|---|---|---|
| **Code** | `fig, ax = plt.subplots()`<br>`ax.scatter(df["x"], df["y"])` | `sns.scatterplot(data=df, x="x", y="y", hue="group")` | `px.scatter(df, x="x", y="y", color="group")` |
| **Strength** | Full control of every element | Grouping and statistics from column names | Interaction and sharing |
| **Typical use** | Publication figures, custom layouts | Exploring and comparing groups | Exploring details, sharing online |
| **Output** | Static image (PNG, SVG, PDF) | Static image (it is Matplotlib) | Interactive HTML |

These are starting points, not strict rules: you can combine them, as we did with Seaborn and Matplotlib.

### Before you share a plot, ask the five questions from the slides

1. **Show:** Is this the right chart for the question and the data?
2. **Explain:** Are the message, quantities, units, and source clear?
3. **Trust:** Could the axes, aggregation, or ordering mislead? (Remember Simpson's paradox.)
4. **Include:** Is it readable, and usable without relying on color alone?
5. **Focus:** Can I remove clutter without removing evidence?

### Learn more

- **Matplotlib:** [Quick start](https://matplotlib.org/stable/users/explain/quick_start.html) · [Cheatsheets](https://matplotlib.org/cheatsheets/) · [Gallery](https://matplotlib.org/stable/gallery/index.html)
- **Seaborn:** [Tutorial](https://seaborn.pydata.org/tutorial.html) · [Gallery](https://seaborn.pydata.org/examples/index.html)
- **Plotly:** [Plotly Express](https://plotly.com/python/plotly-express/) · [Gallery](https://plotly.com/python/)
- **Design:** [Fundamentals of Data Visualization](https://clauswilke.com/dataviz/) (free online book) · [Python Graph Gallery](https://python-graph-gallery.com/)

---
## Extra: more chart types for more questions

If we have time, here are chart types we have not used yet. Each example starts with a question, says why the chart fits it, and needs only a few lines of code. The examples are independent: run any of them.

| Question | Chart | Library |
|---|---|---|
| What share of the whole is each part? | Pie chart | Matplotlib |
| What is the average and spread of each group? | Dot and error bar | Matplotlib |
| Can I show two views in one figure? | Side-by-side panels | Matplotlib |
| How many are in each category? | Count plot | Seaborn |
| How did something change over time? | Line chart | Seaborn |
| Which measurements move together? | Heatmap | Seaborn |
| Does a pattern look the same in each group? | Small multiples | Seaborn |
| How do all the measurements relate? | Pair plot | Seaborn |
| How do groups split into smaller groups? | Sunburst | Plotly |
| Do three measurements together separate the groups? | 3D scatter | Plotly |
| Where do the same individuals fall across many measurements? | Scatter matrix | Plotly |

### More Matplotlib charts

#### Pie chart

**Question:** What share of the measured penguins belongs to each species?

A pie shows parts of one whole. It works for a few clearly different parts, like these three; for many parts or similar sizes, use a sorted bar chart.

In [ ]:
counts = penguins["species"].value_counts()    # number of penguins per species

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(counts.values, labels=counts.index, autopct="%.0f%%")   # autopct writes the percentages
ax.set_title("Share of measured penguins by species")
plt.show()

#### Dot and error bar

**Question:** What is the average body mass of each species, and how much does it vary?

A dot marks each mean and a line shows the spread, here one standard deviation (SD). This is common in papers: always say what the error bars represent. Position, not bar length, shows the value, so the axis does not need to start at zero.

In [ ]:
stats = penguins.groupby("species")["body_mass_g"].agg(["mean", "std"])   # mean and SD per species

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(stats.index, stats["mean"], yerr=stats["std"], fmt="o", capsize=6)   # fmt="o": dots only
ax.set_ylabel("Body mass (g), mean ± 1 SD")
ax.set_title("Average body mass and spread within each species")
plt.show()

#### Side-by-side panels

**Question:** How are flipper lengths distributed, and how do they relate to body mass? We want both in one figure for a paper.

`plt.subplots(1, 2)` creates one row with two axes, so each panel can hold a different chart. This is why the function is called `subplots`.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))   # 1 row, 2 columns of axes

ax1.hist(penguins["flipper_length_mm"], bins=15, color="teal", edgecolor="white")
ax1.set_xlabel("Flipper length (mm)")
ax1.set_ylabel("Number of penguins")
ax1.set_title("A. Distribution of flipper length")

ax2.scatter(penguins["flipper_length_mm"], penguins["body_mass_g"], s=15, alpha=0.5, color="teal")
ax2.set_xlabel("Flipper length (mm)")
ax2.set_ylabel("Body mass (g)")
ax2.set_title("B. Flipper length and body mass")
plt.show()

### More Seaborn charts

#### Count plot

**Question:** How many penguins of each species were measured on each island?

`sns.countplot` counts the rows for you; no summary table needed. *Look for:* which species appear on which island.

In [ ]:
ax = sns.countplot(data=penguins, x="island", hue="species", palette="colorblind")
ax.set_xlabel("Island")
ax.set_ylabel("Number of penguins")
plt.show()

#### Line chart

**Question:** Did the average body mass of each species change between 2007 and 2009?

Lines show change along an ordered axis such as time. `sns.lineplot` calculates each year's mean and shades its 95% confidence interval. Each year measured different penguins, so this compares yearly samples, not the same birds growing.

In [ ]:
ax = sns.lineplot(data=penguins, x="year", y="body_mass_g", hue="species",
                  marker="o", palette="colorblind")
ax.set_xticks([2007, 2008, 2009])    # show whole years only
ax.set_xlabel("Year")
ax.set_ylabel("Average body mass (g)")
plt.show()

#### Heatmap

**Question:** Which body measurements tend to increase together?

A heatmap colors a table of numbers. Correlations run from -1 to +1 around a meaningful middle (0), so we use a **diverging** palette centered at 0, as in the slides. *Look for:* bill depth. Remember 2.2: these correlations pool all species.

In [ ]:
measurements = penguins[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]]
correlations = measurements.corr()    # correlation between every pair of columns

ax = sns.heatmap(correlations, annot=True, cmap="vlag", vmin=-1, vmax=1)   # annot writes the numbers
ax.set_title("Correlation between measurements (all species pooled)")
plt.show()

#### Small multiples

**Question:** Within each species, how do females and males compare in flipper length and body mass?

`col="species"` draws one panel per species. The panels share their axes, so they are directly comparable, as recommended in the slides.

In [ ]:
g = sns.relplot(data=penguins, x="flipper_length_mm", y="body_mass_g",
                col="species", hue="sex", palette="colorblind", height=3.5)
g.set_axis_labels("Flipper length (mm)", "Body mass (g)")
plt.show()

#### Pair plot

**Question:** Which pair of measurements separates the three species best?

`sns.pairplot` draws a scatter plot for every pair of number columns, with each column's distribution on the diagonal: a quick overview of a new dataset.

In [ ]:
sns.pairplot(penguins.drop(columns="year"), hue="species", palette="colorblind", height=2)   # text columns are skipped
plt.show()

### More Plotly charts

#### Sunburst

**Question:** How are the measured penguins split by island, then species, then sex?

A sunburst shows groups within groups. Click a slice to zoom into it; click the center to zoom back out. It is good for exploring a hierarchy; for precise comparisons, bars are better.

In [ ]:
fig = px.sunburst(penguins, path=["island", "species", "sex"],
                  title="Penguins by island, species, and sex")
fig.show()

#### 3D scatter

**Question:** Do three measurements together separate the species?

Drag to rotate. 3D is hard to read on a printed page, but a view you can rotate can reveal groups.

In [ ]:
fig = px.scatter_3d(penguins, x="bill_length_mm", y="bill_depth_mm", z="flipper_length_mm",
                    color="species", opacity=0.7)
fig.show()

#### Scatter matrix

**Question:** Where do the same penguins fall across all four measurements?

The interactive version of a pair plot. Choose **Box Select** in the toolbar and drag over a few points in one panel: the same penguins are highlighted in every panel. Double-click to clear the selection.

In [ ]:
fig = px.scatter_matrix(penguins, color="species", height=700,
                        dimensions=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"])
fig.update_traces(diagonal_visible=False)    # hide the panels that compare a column with itself
fig.show()